<a href="https://colab.research.google.com/github/SanjaraT/Langchain/blob/main/YouTube%20Chatbot/youtube_rag_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [2]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-huggingface
!pip install -q langchain-ollama
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -U youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 16.6 MB/s eta 0:00:00


# Extract Transcript

In [6]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "hVM8qGRTaOA"

ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch(video_id)

full_text = " ".join(
    [item.text for item in transcript]
)

print(full_text[:1000])

you might have seen machine learning models that take some text and predict whether it is Spam or not similarly a model can analyze a movie review and determine its sentiment whether it's positive or negative let's consider that we are using a neural network to train such models the first rule of training any machine learning model is converting the input into numbers let's take an example of an image in a grayscale image like the one shown on the screen the brightest pixel has a value of one the darkest pixel has a value of zero and Shades of Gray have values between 0 and one this numerical representation makes it simple to process images by converting an image into numerical form based on its pixel values we can use it as input to train our model here a neural network can learn from the pixel value however in this video we are dealing with text so let's see how text data is prepared for machine learning models so we have text and one simple way to convert it into numbers is by assig

# Transcript --> Document

In [7]:
from langchain_core.documents import Document

doc = Document(
    page_content=full_text,
    metadata={"source": "youtube"}
)

docs = [doc]

# Split into Chunks

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

print(len(chunks))

49


# Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

# Vector Store

In [10]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

/tmp/ipykernel_9090/2608617100.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


# Retriever

In [11]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)

# LLM

In [ ]:
from transformers import pipeline
import torch

# Check if CUDA is available and set the device accordingly
device = 0 if torch.cuda.is_available() else -1

pipe = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    max_new_tokens=300,
    device=device # Specify the device for GPU or CPU
)

# Prompt

In [13]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
Answer the question based only on the context.

Context:
{context}

Question:
{question}
""",
    input_variables=[
        "context",
        "question"
    ]
)

# RAG Chain

In [14]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
)

# Query

In [15]:
question = "What are the main topics discussed?"

docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content
    for doc in docs
)

final_prompt = prompt.invoke({
    "context": context,
    "question": question
})

response = pipe(
    str(final_prompt),
    max_new_tokens=200
)

print(response[0]["generated_text"])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


text="\nAnswer the question based only on the context.\n\nContext:\nwe can ask ourselves where was Sarah going and we can easily relate the answer to the word Library similarly consider the word old in the sentence is it referring to Sarah or the library we understand from the sentence that old is describing the library not Sarah also the sentence mentions something was hidden behind the dusty thing we can infer that the hidden object is behind something Dusty which helps us predict what the next word could be when I say the model should understand semantic\n\nhow they help represent words and their positions in the Transformer mind model now we are finally entering the decoder layer and the most important part of the decoder layer the attention mechanism this is a crucial component of large language models and we will cover it in detail in the next video if you are enjoying this series make sure to like and share it with your friends and if you haven't subscribed yet subscribe now to 